In [2]:
import scanpy as sc
import os
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
os.getcwd()

'/home/lixiangyu/zr/Annotate/ANNOTATE_new/10_downstream/Infer_plaque_type/0523_no_Basophil'

# with other unknown

整体流程

```text
第一步：读取包含 stable / unstable / unknown 的 h5ad
        ↓
第二步：用全部已知 stable / unstable 样本生成最终训练用特征：
        - X_basic_celltype_proportion.csv
        - auto_marker_dict.json
        - X_pseudobulk_marker.csv
        - X_combined_basic_plus_pseudobulk.csv
        - y_sample_label_numeric.csv
        ↓
第三步：严格交叉验证：
        每个 fold 里只用 train samples 找 marker，再评估 test samples
        这样避免 marker selection leakage
        ↓
第四步：最终预测 unknown：
        用全部 stable / unstable 样本训练最终模型
        用全部 stable / unstable 找到的 marker_dict 去生成 unknown 特征并预测
```

注意：

- 交叉验证 / 画折线图：不能先用全部样本找 marker，否则测试集标签已经参与 marker 筛选，会虚高。
- 最终训练 / 预测 unknown：可以、也应该使用全部 stable / unstable 样本训练最终模型。
- 所有 human 输出统一保存到 `./output_auto/human/`。


In [2]:
def get_mean_expression(sub_adata, genes):
    if len(genes) == 0:
        return pd.Series(dtype=float)
    sub = sub_adata[:, genes]
    X = sub.X
    mean_expr = np.asarray(X.mean(axis=0)).ravel() if sparse.issparse(X) else np.asarray(X.mean(axis=0)).ravel()
    return pd.Series(mean_expr, index=genes)


def _bh_fdr(pvals):
    """Benjamini-Hochberg FDR correction, no statsmodels dependency."""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    if valid.sum() == 0:
        return qvals

    p = pvals[valid]
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)

    q_valid = np.empty(n, dtype=float)
    q_valid[order] = q
    qvals[valid] = q_valid
    return qvals


def _sample_pseudobulk_matrix(adata_ct, sample_col, label_col):
    """
    对某一个 cell type 的 AnnData，按 sample 聚合成 sample × gene 的平均表达矩阵。
    返回 X_pb, sample_labels, sample_cell_counts。
    """
    obs = adata_ct.obs[[sample_col, label_col]].copy()
    obs[sample_col] = obs[sample_col].astype(str)
    obs[label_col] = obs[label_col].astype(str)

    sample_ids = sorted(obs[sample_col].unique().tolist())
    rows = []
    labels = []
    n_cells = []

    for sid in sample_ids:
        mask = (obs[sample_col].values == sid)
        sub = adata_ct[mask]
        X = sub.X
        mean_expr = np.asarray(X.mean(axis=0)).ravel() if sparse.issparse(X) else np.asarray(X.mean(axis=0)).ravel()
        rows.append(mean_expr)

        lab = obs.loc[mask, label_col].unique().tolist()
        if len(lab) != 1:
            raise ValueError(f"sample {sid} 在同一个 cell type 中有多个 label: {lab}")
        labels.append(lab[0])
        n_cells.append(int(mask.sum()))

    X_pb = pd.DataFrame(np.vstack(rows), index=sample_ids, columns=adata_ct.var_names.astype(str))
    sample_labels = pd.Series(labels, index=sample_ids, name="label")
    sample_cell_counts = pd.Series(n_cells, index=sample_ids, name="n_cells")
    return X_pb, sample_labels, sample_cell_counts

##如果某个 cell type 在 stable 或 unstable 里样本太少，它就跳过，不乱做差异分析。代码里这一段是在检查细胞数和 sample 数
def auto_find_markers_by_celltype(
    adata,
    sample_col,
    celltype_col,
    label_col,
    candidate_celltypes,
    top_n=20,##每个 cell type 最多选 20 个 marker genes
    min_cells_per_group=30,##stable/unstable里的细胞类型的细胞数>30
    min_samples_per_group=3,##某个 cell type 在 stable/unstable组至少来自 3 个 sample
    pval_adj_cutoff=0.05,##只保留 FDR 校正后 p 值小于 0.05 的基因
    min_logfc=0.0,##这个gene：mean_unstable(在所有unstable的样本中平均表达) - mean_stable > 0
    method="mannwhitneyu",
    output_dir=None,
):
    """
    方案 B：用户指定候选 cell type；每个 cell type 内部自动找 unstable vs stable marker genes。

    重要：这里做的是 sample-level pseudobulk DE，不再使用 cell-level rank_genes_groups。
    对每个 cell type：
      1. 每个 sample 内聚合该 cell type 的平均表达，得到 sample × gene 矩阵；
      2. 在 sample 层面比较 unstable samples vs stable samples；
      3. FDR 校正后筛选 unstable 方向更高的 genes。

    输出：
      - auto_marker_table.csv
      - auto_marker_dict.json
      - auto_marker_skipped_celltypes.csv
    """
    import json
    from scipy.stats import mannwhitneyu, ttest_ind

    marker_dict = {}
    marker_rows = []
    skipped_rows = []

    adata_use = adata[adata.obs[label_col].isin(["stable", "unstable"])].copy()

    print("\n===== auto marker discovery: sample-level pseudobulk unstable vs stable within each cell type =====")
    print("candidate_celltypes:", candidate_celltypes)
    print("DE method:", method)

    for ct in candidate_celltypes:
        ct_mask = adata_use.obs[celltype_col].astype(str) == str(ct)
        adata_ct = adata_use[ct_mask].copy()

        if adata_ct.n_obs == 0:
            msg = "cell type not found"
            print(f"Skip {ct}: {msg}")
            skipped_rows.append({"cell_type": ct, "reason": msg})
            continue

        n_stable = int((adata_ct.obs[label_col] == "stable").sum())
        n_unstable = int((adata_ct.obs[label_col] == "unstable").sum())
        stable_samples = int(adata_ct.obs.loc[adata_ct.obs[label_col] == "stable", sample_col].astype(str).nunique())
        unstable_samples = int(adata_ct.obs.loc[adata_ct.obs[label_col] == "unstable", sample_col].astype(str).nunique())

        print(
            f"\n{ct}: n_cells stable={n_stable}, unstable={n_unstable}; "
            f"n_samples stable={stable_samples}, unstable={unstable_samples}"
        )

        if n_stable < min_cells_per_group or n_unstable < min_cells_per_group:
            msg = f"not enough cells: stable={n_stable}, unstable={n_unstable}"
            print(f"Skip {ct}: {msg}")
            skipped_rows.append({"cell_type": ct, "reason": msg})
            continue

        if stable_samples < min_samples_per_group or unstable_samples < min_samples_per_group:
            msg = f"not enough samples: stable={stable_samples}, unstable={unstable_samples}"
            print(f"Skip {ct}: {msg}")
            skipped_rows.append({"cell_type": ct, "reason": msg})
            continue

        try:
            X_pb_ct, sample_labels_ct, sample_cell_counts_ct = _sample_pseudobulk_matrix(
                adata_ct=adata_ct,
                sample_col=sample_col,
                label_col=label_col,
            )
        except Exception as e:
            msg = f"pseudobulk aggregation failed: {repr(e)}"
            print(f"Skip {ct}: {msg}")
            skipped_rows.append({"cell_type": ct, "reason": msg})
            continue

        stable_mask = sample_labels_ct.values == "stable"
        unstable_mask = sample_labels_ct.values == "unstable"
        X_stable = X_pb_ct.loc[stable_mask].to_numpy(dtype=float)
        X_unstable = X_pb_ct.loc[unstable_mask].to_numpy(dtype=float)

        mean_stable = np.nanmean(X_stable, axis=0)
        mean_unstable = np.nanmean(X_unstable, axis=0)
        mean_diff = mean_unstable - mean_stable

        pvals = np.full(X_pb_ct.shape[1], np.nan, dtype=float)
        stats = np.full(X_pb_ct.shape[1], np.nan, dtype=float)

        for j in range(X_pb_ct.shape[1]):
            a = X_unstable[:, j]
            b = X_stable[:, j]
            a = a[np.isfinite(a)]
            b = b[np.isfinite(b)]
            if len(a) < 2 or len(b) < 2:
                continue
            if np.nanstd(a) == 0 and np.nanstd(b) == 0 and np.nanmean(a) == np.nanmean(b):
                pvals[j] = 1.0
                stats[j] = 0.0
                continue
            try:
                if method.lower() in ["mannwhitney", "mannwhitneyu", "wilcoxon"]:
                    res = mannwhitneyu(a, b, alternative="two-sided")
                    pvals[j] = float(res.pvalue)
                    stats[j] = float(res.statistic)
                elif method.lower() in ["welch", "welch_ttest", "ttest"]:
                    res = ttest_ind(a, b, equal_var=False, nan_policy="omit")
                    pvals[j] = float(res.pvalue)
                    stats[j] = float(res.statistic)
                else:
                    raise ValueError(f"Unsupported method: {method}")
            except Exception:
                pvals[j] = np.nan
                stats[j] = np.nan

        pvals_adj = _bh_fdr(pvals)

        df_marker = pd.DataFrame({
            "names": X_pb_ct.columns.astype(str),
            "scores": stats,
            # 这里保留 logfoldchanges 这个列名，是为了兼容后续表格；
            # 对 log-normalized scRNA 表达矩阵，它代表 unstable - stable 的 pseudobulk 平均表达差。
            "logfoldchanges": mean_diff,
            "mean_unstable": mean_unstable,
            "mean_stable": mean_stable,
            "pvals": pvals,
            "pvals_adj": pvals_adj,
        })

        df_sig = df_marker[
            (df_marker["pvals_adj"] < pval_adj_cutoff) &
            (df_marker["logfoldchanges"] > min_logfc)
        ].copy()
        selection_rule = (
            f"sample-level pseudobulk {method}: pvals_adj < {pval_adj_cutoff} "
            f"and unstable_minus_stable > {min_logfc}"
        )

        if df_sig.empty:
            df_sig = df_marker[df_marker["logfoldchanges"] > min_logfc].copy()
            selection_rule = (
                f"fallback sample-level pseudobulk {method}: "
                f"unstable_minus_stable > {min_logfc}, sorted by p-value/effect"
            )

        if df_sig.empty:
            msg = "no positive unstable markers at sample-level pseudobulk"
            print(f"Skip {ct}: {msg}")
            skipped_rows.append({"cell_type": ct, "reason": msg})
            continue

        df_sig["abs_effect"] = df_sig["logfoldchanges"].abs()
        df_sig = df_sig.sort_values(["pvals_adj", "pvals", "abs_effect"], ascending=[True, True, False])

        selected = df_sig.head(top_n).copy()
        genes = selected["names"].astype(str).tolist()
        marker_dict[ct] = genes

        print(f"Selected {len(genes)} sample-level pseudobulk markers for {ct} using rule: {selection_rule}")
        print(genes)

        selected.insert(0, "rank", range(1, selected.shape[0] + 1))
        selected.insert(0, "selection_rule", selection_rule)
        selected.insert(0, "de_unit", "sample_pseudobulk")
        selected.insert(0, "n_unstable_samples", unstable_samples)
        selected.insert(0, "n_stable_samples", stable_samples)
        selected.insert(0, "n_unstable_cells", n_unstable)
        selected.insert(0, "n_stable_cells", n_stable)
        selected.insert(0, "cell_type", ct)
        marker_rows.append(selected)

    if len(marker_dict) == 0:
        raise ValueError("没有找到任何可用的自动 marker。请降低 min_cells_per_group / min_samples_per_group，或检查 cell type 名称。")

    marker_table = pd.concat(marker_rows, axis=0, ignore_index=True) if marker_rows else pd.DataFrame()
    skipped_table = pd.DataFrame(skipped_rows)

    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        marker_table_path = os.path.join(output_dir, "auto_marker_table.csv")
        marker_dict_path = os.path.join(output_dir, "auto_marker_dict.json")
        skipped_path = os.path.join(output_dir, "auto_marker_skipped_celltypes.csv")

        marker_table.to_csv(marker_table_path, index=False)
        with open(marker_dict_path, "w") as f:
            json.dump(marker_dict, f, indent=2)
        skipped_table.to_csv(skipped_path, index=False)

        print(f"\nSaved auto marker table: {marker_table_path}")
        print(f"Saved auto marker dict: {marker_dict_path}")
        print(f"Saved skipped cell types: {skipped_path}")

    return marker_dict, marker_table, skipped_table


def load_marker_dict(marker_json_path):
    """读取 prepare input 阶段保存的 auto_marker_dict.json，保证预测新数据时使用同一套 marker。"""
    import json
    with open(marker_json_path, "r") as f:
        marker_dict = json.load(f)
    return {str(ct): [str(g) for g in genes] for ct, genes in marker_dict.items()}


def build_pseudobulk_features(adata_in, marker_dict, sample_col, celltype_col):
    all_genes = set(adata_in.var_names)
    filtered_marker_dict, missing_marker_report = {}, {}
    for ct, genes in marker_dict.items():
        present = [g for g in genes if g in all_genes]
        missing = [g for g in genes if g not in all_genes]
        filtered_marker_dict[ct] = present
        missing_marker_report[ct] = missing
    print("\n===== marker presence check =====")
    for ct in marker_dict:
        print(f"{ct}:")
        print("  present:", filtered_marker_dict[ct])
        print("  missing:", missing_marker_report[ct])
    filtered_marker_dict = {ct: genes for ct, genes in filtered_marker_dict.items() if len(genes) > 0}
    all_celltypes = set(adata_in.obs[celltype_col].astype(str).unique())
    print("\n===== cell types in adata =====")
    print(sorted(list(all_celltypes))[:50])
    valid_celltypes = [ct for ct in filtered_marker_dict if ct in all_celltypes]
    invalid_celltypes = [ct for ct in filtered_marker_dict if ct not in all_celltypes]
    if len(invalid_celltypes) > 0:
        print("\n这些 marker_dict 里的 cell type 不在 adata.obs 里，请检查命名：")
        print(invalid_celltypes)
    filtered_marker_dict = {ct: filtered_marker_dict[ct] for ct in valid_celltypes}
    if len(filtered_marker_dict) == 0:
        raise ValueError("没有任何有效的 cell type + marker 组合。")
    feature_rows = []
    sample_ids = adata_in.obs[sample_col].astype(str).unique().tolist()
    for sample_id in sample_ids:
        sample_mask = adata_in.obs[sample_col].astype(str) == str(sample_id)
        adata_sample = adata_in[sample_mask].copy()
        row_dict = {"sample": sample_id}
        for ct, genes in filtered_marker_dict.items():
            ct_mask = adata_sample.obs[celltype_col].astype(str) == ct
            n_cells = int(ct_mask.sum())
            if n_cells == 0:
                for g in genes:
                    row_dict[f"pb__{ct}__{g}"] = 0.0
                row_dict[f"pb_ncells__{ct}"] = 0
                continue
            adata_sub = adata_sample[ct_mask].copy()
            mean_expr = get_mean_expression(adata_sub, genes)
            for g in genes:
                row_dict[f"pb__{ct}__{g}"] = float(mean_expr[g])
            row_dict[f"pb_ncells__{ct}"] = n_cells
        feature_rows.append(row_dict)
    X_pb = pd.DataFrame(feature_rows).set_index("sample").sort_index()
    return X_pb

## prepare input

In [3]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

# ========================= 参数 =========================
H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_allhuman/scPoli_concat_level3_marker_all_metadata.h5ad"
OUT_DIR = "./output_auto/human"
os.makedirs(OUT_DIR, exist_ok=True)

SAMPLE_COL = "sample"
# CELLTYPE_COL = "cell_type_level1"
CELLTYPE_COL = "cell_type_level1_corrected"##应该用cell_type_level1_corrected
LABEL_RAW_COL = "Plaque_type"
LABEL_COL = "Plaque_type_clean"

TRAIN_X_BASIC_PATH = f"{OUT_DIR}/X_basic_celltype_proportion.csv"
TRAIN_X_PB_PATH = f"{OUT_DIR}/X_pseudobulk_marker.csv"
TRAIN_X_COMBINED_PATH = f"{OUT_DIR}/X_combined_basic_plus_pseudobulk.csv"
Y_LABEL_PATH = f"{OUT_DIR}/y_sample_label.csv"
Y_NUM_PATH = f"{OUT_DIR}/y_sample_label_numeric.csv"

OUT_X_BASIC_UNKNOWN = f"{OUT_DIR}/X_basic_unknown_celltype_proportion.csv"
OUT_X_PB_UNKNOWN = f"{OUT_DIR}/X_pseudobulk_marker_unknown.csv"
OUT_X_UNKNOWN_COMBINED = f"{OUT_DIR}/X_unknown_combined_basic_plus_pseudobulk.csv"

# 方案 B：指定候选 cell type，marker genes 由代码在每个 cell type 内自动寻找
CANDIDATE_CELLTYPES = [
    "Macrophage",
    "Monocyte",
    "Dendritic cell",
    "Smooth muscle cell",
    "Endothelial cell",
    "Fibroblast",
    "T cell",
]
AUTO_MARKER_TOP_N = 20
MIN_CELLS_PER_GROUP = 30
MIN_SAMPLES_PER_GROUP = 3
PVAL_ADJ_CUTOFF = 0.05
MIN_LOGFC = 0.0

AUTO_MARKER_TABLE_PATH = f"{OUT_DIR}/auto_marker_table.csv"
AUTO_MARKER_DICT_PATH = f"{OUT_DIR}/auto_marker_dict.json"
AUTO_MARKER_SKIPPED_PATH = f"{OUT_DIR}/auto_marker_skipped_celltypes.csv"


# ========================= 1. 读取和预处理 h5ad =========================
print("Reading h5ad...")
adata = sc.read_h5ad(H5AD_PATH)
adata = adata[adata.obs[CELLTYPE_COL] != "unknown"].copy()
adata.var["original_feature_id"] = adata.var_names.astype(str)
adata.var_names = adata.var["original_gene_names"].astype(str)
adata.var_names_make_unique()
adata.obs[LABEL_COL] = adata.obs[LABEL_RAW_COL].astype(str).str.strip().str.lower()

adata.obs[LABEL_COL] = adata.obs[LABEL_COL].replace({
    "nan": "unknown",
    "na": "unknown",
    "none": "unknown",
    "": "unknown",
    "unknown": "unknown",
    "Unknown": "unknown",
})
adata.obs[LABEL_COL] = adata.obs[LABEL_COL].fillna("unknown")

required_cols = [SAMPLE_COL, CELLTYPE_COL, LABEL_COL]
missing_cols = [c for c in required_cols if c not in adata.obs.columns]
if len(missing_cols) > 0:
    raise ValueError(f"adata.obs 缺少这些列: {missing_cols}")

print("adata shape after filtering unknown cell type:", adata.shape)
print("sample label distribution:")
print(adata.obs[LABEL_COL].value_counts(dropna=False))

# ========================= 2. 生成训练集 X_basic 和 y =========================
print("\nBuilding training X_basic and y...")
obs = adata.obs[[SAMPLE_COL, CELLTYPE_COL, LABEL_COL]].copy()
obs = obs.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])
train_obs = obs[obs[LABEL_COL].isin(["stable", "unstable"])].copy()

print("训练细胞数:", train_obs.shape[0])
print("训练标签分布:")
print(train_obs[LABEL_COL].value_counts())

cell_count = pd.crosstab(train_obs[SAMPLE_COL], train_obs[CELLTYPE_COL])
cell_prop = cell_count.div(cell_count.sum(axis=1), axis=0)
cell_prop.columns = [f"prop__{c}" for c in cell_prop.columns]

sample_label = train_obs[[SAMPLE_COL, LABEL_COL]].drop_duplicates().groupby(SAMPLE_COL, observed=True)[LABEL_COL].agg(lambda x: x.iloc[0] if len(set(x)) == 1 else "conflict")
conflict_samples = sample_label[sample_label == "conflict"].index.tolist()
if len(conflict_samples) > 0:
    print("仍然冲突的 sample:", conflict_samples[:20])
    raise ValueError("还有 sample 存在多个标签，请进一步检查。")

sample_label = sample_label.rename("label")
common_samples = cell_prop.index.intersection(sample_label.index)
X_basic = cell_prop.loc[common_samples].copy().sort_index()
y = sample_label.loc[common_samples].copy().sort_index()
y_num = y.map({"stable": 0, "unstable": 1})

print("X_basic shape:", X_basic.shape)
print("y shape:", y_num.shape)
print("y label counts:")
print(y.value_counts())

X_basic.to_csv(TRAIN_X_BASIC_PATH)
y.to_csv(Y_LABEL_PATH, header=True)
y_num.to_csv(Y_NUM_PATH, header=True)
print(f"Saved: {TRAIN_X_BASIC_PATH}")
print(f"Saved: {Y_LABEL_PATH}")
print(f"Saved: {Y_NUM_PATH}")

# ========================= 3. 生成训练集 X_pseudobulk 和 X_combined =========================
print("\nBuilding training X_pseudobulk...")
adata_train = adata[adata.obs[LABEL_COL].isin(["stable", "unstable"])].copy()

# 自动找 marker：每个指定 cell type 内部先按 sample 做 pseudobulk，再做 unstable vs stable 差异分析
marker_dict, auto_marker_table, auto_marker_skipped = auto_find_markers_by_celltype(
    adata=adata_train,
    sample_col=SAMPLE_COL,
    celltype_col=CELLTYPE_COL,
    label_col=LABEL_COL,
    candidate_celltypes=CANDIDATE_CELLTYPES,
    top_n=AUTO_MARKER_TOP_N,
    min_cells_per_group=MIN_CELLS_PER_GROUP,
    min_samples_per_group=MIN_SAMPLES_PER_GROUP,
    pval_adj_cutoff=PVAL_ADJ_CUTOFF,
    min_logfc=MIN_LOGFC,
    method="mannwhitneyu",
    output_dir=OUT_DIR,
)

X_pb = build_pseudobulk_features(adata_train, marker_dict, SAMPLE_COL, CELLTYPE_COL)
X_pb.to_csv(TRAIN_X_PB_PATH)
print("X_pb shape:", X_pb.shape)
print(f"Saved: {TRAIN_X_PB_PATH}")

common_samples = X_basic.index.intersection(X_pb.index)
X_combined = pd.concat([X_basic.loc[common_samples], X_pb.loc[common_samples]], axis=1)
X_combined.to_csv(TRAIN_X_COMBINED_PATH)
print("X_combined shape:", X_combined.shape)
print(f"Saved: {TRAIN_X_COMBINED_PATH}")

# ========================= 4. 生成 unknown 样本 X_basic_unknown =========================
print("\nBuilding unknown features...")
adata_unknown = adata[adata.obs[LABEL_COL].isin(["unknown"])].copy()
print("adata_unknown shape:", adata_unknown.shape)
print("unknown label counts:")
print(adata_unknown.obs[LABEL_COL].value_counts())

if adata_unknown.n_obs == 0:
    raise ValueError("没有找到标签为 unknown 的样本。")

obs_unknown = adata_unknown.obs[[SAMPLE_COL, CELLTYPE_COL, LABEL_COL]].copy()
obs_unknown = obs_unknown.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])
cell_count_unknown = pd.crosstab(obs_unknown[SAMPLE_COL], obs_unknown[CELLTYPE_COL])
cell_prop_unknown = cell_count_unknown.div(cell_count_unknown.sum(axis=1), axis=0)
cell_prop_unknown.columns = [f"prop__{c}" for c in cell_prop_unknown.columns]

X_basic_unknown = cell_prop_unknown.copy()
X_basic_unknown = X_basic_unknown.reindex(columns=X_basic.columns, fill_value=0)
X_basic_unknown.to_csv(OUT_X_BASIC_UNKNOWN)
print("X_basic_unknown shape:", X_basic_unknown.shape)
print(f"Saved: {OUT_X_BASIC_UNKNOWN}")

# ========================= 5. 生成 unknown 样本 X_pseudobulk 和 X_unknown_combined =========================
print("\nBuilding unknown X_pseudobulk...")
X_pb_unknown = build_pseudobulk_features(adata_unknown, marker_dict, SAMPLE_COL, CELLTYPE_COL)
X_pb_unknown = X_pb_unknown.reindex(columns=X_pb.columns, fill_value=0)
X_pb_unknown.to_csv(OUT_X_PB_UNKNOWN)
print("X_pb_unknown shape:", X_pb_unknown.shape)
print(f"Saved: {OUT_X_PB_UNKNOWN}")

common_unknown_samples = X_basic_unknown.index.intersection(X_pb_unknown.index)
X_unknown_combined = pd.concat([X_basic_unknown.loc[common_unknown_samples], X_pb_unknown.loc[common_unknown_samples]], axis=1)
X_unknown_combined = X_unknown_combined.reindex(columns=X_combined.columns, fill_value=0)
X_unknown_combined.to_csv(OUT_X_UNKNOWN_COMBINED)
print("X_unknown_combined shape:", X_unknown_combined.shape)
print(f"Saved: {OUT_X_UNKNOWN_COMBINED}")

print("\nDone.")

Reading h5ad...
adata shape after filtering unknown cell type: (1004247, 28868)
sample label distribution:
Plaque_type_clean
unknown     790610
unstable    152880
stable       60757
Name: count, dtype: int64

Building training X_basic and y...
训练细胞数: 213637
训练标签分布:
Plaque_type_clean
unstable    152880
stable       60757
Name: count, dtype: int64
X_basic shape: (23, 13)
y shape: (23,)
y label counts:
label
unstable    16
stable       7
Name: count, dtype: int64
Saved: ./output_auto/human/X_basic_celltype_proportion.csv
Saved: ./output_auto/human/y_sample_label.csv
Saved: ./output_auto/human/y_sample_label_numeric.csv

Building training X_pseudobulk...

===== auto marker discovery: sample-level pseudobulk unstable vs stable within each cell type =====
candidate_celltypes: ['Macrophage', 'Monocyte', 'Dendritic cell', 'Smooth muscle cell', 'Endothelial cell', 'Fibroblast', 'T cell']
DE method: mannwhitneyu

Macrophage: n_cells stable=9740, unstable=10085; n_samples stable=7, unstable=14
Se

严格折线图评估：每个 fold 内重新找 marker，避免信息泄漏

这一步只用于评估模型泛化能力，不影响最终模型训练。

严格流程：

```text
每个 train_size、每个 fold：
    1. 先切分 train samples / test samples
    2. 只用 train samples 做 sample-level pseudobulk DE 找 marker
    3. 用该 fold 的 marker 分别生成 train/test pseudobulk 特征
    4. 训练 LogisticRegression
    5. 在完全没参与 marker 选择的 test samples 上评估
```

最终预测 unknown 时仍然会使用 prepare input 阶段保存的全部 stable / unstable 训练特征。


# 交叉验证

## 折线图--随机划分 5 次

In [ ]:
# ============================================================
# 严格交叉验证：fold-wise marker selection，避免 feature selection leakage
# ============================================================
# 说明：
# - 这一格只用于评估模型是否真的能泛化。
# - 每个 fold 都只在 train samples 里找 marker。
# - test samples 不参与 marker discovery。
# - 最终预测 unknown 时，仍然使用全部 stable / unstable 样本训练最终模型。

import os
import json
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# -------------------------
# 1. 参数
# -------------------------
OUT_DIR = "./output_auto/human"
os.makedirs(OUT_DIR, exist_ok=True)

train_sizes = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
n_splits = 5
threshold = 0.5
random_state = 42

SAMPLE_COL = "sample"
# CELLTYPE_COL = "cell_type_level1"
CELLTYPE_COL = "cell_type_level1_corrected"##应该用cell_type_level1_corrected
LABEL_RAW_COL = "Plaque_type"
LABEL_COL = "Plaque_type_clean"

CV_SUMMARY_OUT = f"{OUT_DIR}/lr_train_size_cv_summary_foldwise_marker.csv"
CV_PRED_OUT = f"{OUT_DIR}/lr_train_size_cv_all_fold_predictions_foldwise_marker.csv"
CV_MARKER_COUNT_OUT = f"{OUT_DIR}/lr_train_size_cv_marker_counts_foldwise_marker.csv"
CANDIDATE_CELLTYPES = [
    "Macrophage",
    "Monocyte",
    "Dendritic cell",
    "Smooth muscle cell",
    "Endothelial cell",
    "Fibroblast",
    "T cell",
]
AUTO_MARKER_TOP_N = 20
MIN_CELLS_PER_GROUP = 30
MIN_SAMPLES_PER_GROUP = 3
PVAL_ADJ_CUTOFF = 0.05
MIN_LOGFC = 0.0
# 如果某些 fold 因为训练样本太少导致某个 cell type 找不到显著 marker，
# 可以适当降低 MIN_SAMPLES_PER_GROUP 或 MIN_CELLS_PER_GROUP。
# 这里默认沿用 prepare input 的参数。
print("Strict CV output dir:", OUT_DIR)
print("n_splits:", n_splits)
print("threshold:", threshold)

# -------------------------
# 2. 读取 y，并准备 labeled adata
# -------------------------
y = pd.read_csv(f"{OUT_DIR}/y_sample_label_numeric.csv", index_col=0).iloc[:, 0]
y.index = y.index.astype(str)
y = y.sort_index()
print("Reading h5ad...")
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_allhuman/scPoli_concat_level3_marker_all_metadata.h5ad")
adata = adata[adata.obs[CELLTYPE_COL] != "unknown"].copy()
adata.var["original_feature_id"] = adata.var_names.astype(str)
adata.var_names = adata.var["original_gene_names"].astype(str)
adata.var_names_make_unique()
adata.obs[LABEL_COL] = adata.obs[LABEL_RAW_COL].astype(str).str.strip().str.lower()

adata.obs[LABEL_COL] = adata.obs[LABEL_COL].replace({
    "nan": "unknown",
    "na": "unknown",
    "none": "unknown",
    "": "unknown",
    "unknown": "unknown",
    "Unknown": "unknown",
})
adata.obs[LABEL_COL] = adata.obs[LABEL_COL].fillna("unknown")

required_cols = [SAMPLE_COL, CELLTYPE_COL, LABEL_COL]
missing_cols = [c for c in required_cols if c not in adata.obs.columns]
if len(missing_cols) > 0:
    raise ValueError(f"adata.obs 缺少这些列: {missing_cols}")

print("adata shape after filtering unknown cell type:", adata.shape)
print("sample label distribution:")

print(adata.obs[LABEL_COL].value_counts(dropna=False))
if "adata" not in globals():
    raise RuntimeError("请先运行前面的 prepare input cell，让变量 adata 存在于当前 notebook 环境中。")

adata_labeled = adata[adata.obs[LABEL_COL].isin(["stable", "unstable"])].copy()
adata_labeled.obs[SAMPLE_COL] = adata_labeled.obs[SAMPLE_COL].astype(str)

# 只保留 adata 里确实存在的样本
available_samples = set(adata_labeled.obs[SAMPLE_COL].unique().astype(str))
y = y.loc[[sid for sid in y.index if sid in available_samples]]

sample_ids = y.index.to_numpy()

print("Number of labeled samples:", len(sample_ids))
print("y distribution:")
print(y.value_counts())

# -------------------------
# 3. helper：按 sample 生成 cell-type proportion 特征
# -------------------------
def build_basic_features_for_samples(adata_in, sample_ids, sample_col, celltype_col, ref_columns=None):
    sample_ids = [str(s) for s in sample_ids]
    obs = adata_in.obs[[sample_col, celltype_col]].copy()
    obs[sample_col] = obs[sample_col].astype(str)
    obs[celltype_col] = obs[celltype_col].astype(str)
    obs = obs[obs[sample_col].isin(sample_ids)].copy()

    if obs.empty:
        X_basic = pd.DataFrame(index=sample_ids)
    else:
        cell_count = pd.crosstab(obs[sample_col], obs[celltype_col])
        cell_prop = cell_count.div(cell_count.sum(axis=1), axis=0)
        cell_prop.columns = [f"prop__{c}" for c in cell_prop.columns]
        X_basic = cell_prop.copy()

    X_basic.index = X_basic.index.astype(str)
    X_basic = X_basic.reindex(index=sample_ids, fill_value=0)

    if ref_columns is not None:
        X_basic = X_basic.reindex(columns=ref_columns, fill_value=0)

    return X_basic


def subset_adata_by_samples(adata_in, sample_ids, sample_col):
    sample_ids = set([str(s) for s in sample_ids])
    mask = adata_in.obs[sample_col].astype(str).isin(sample_ids)
    return adata_in[mask].copy()


summary_rows = []
all_fold_predictions = []
marker_count_rows = []

# -------------------------
# 4. 严格 CV
# -------------------------
for train_size in train_sizes:
    test_size = 1 - train_size

    print("\n" + "=" * 90)
    print(f"Strict CV: train_size = {int(train_size * 100)}%, test_size = {int(test_size * 100)}%")
    print("=" * 90)

    splitter = StratifiedShuffleSplit(
        n_splits=n_splits,
        train_size=train_size,
        test_size=test_size,
        random_state=random_state
    )

    fold_metrics = []

    for fold_id, (train_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y)), y.values), start=1):
        train_samples = sample_ids[train_idx].astype(str).tolist()
        test_samples = sample_ids[test_idx].astype(str).tolist()

        y_train = y.loc[train_samples]
        y_test = y.loc[test_samples]

        adata_train_fold = subset_adata_by_samples(adata_labeled, train_samples, SAMPLE_COL)
        adata_test_fold = subset_adata_by_samples(adata_labeled, test_samples, SAMPLE_COL)

        print(f"\nFold {fold_id}: n_train={len(train_samples)}, n_test={len(test_samples)}")

        # 4.1 只在 train samples 内找 marker
        try:
            marker_dict_fold, marker_table_fold, skipped_table_fold = auto_find_markers_by_celltype(
                adata=adata_train_fold,
                sample_col=SAMPLE_COL,
                celltype_col=CELLTYPE_COL,
                label_col=LABEL_COL,
                candidate_celltypes=CANDIDATE_CELLTYPES,
                top_n=AUTO_MARKER_TOP_N,
                min_cells_per_group=MIN_CELLS_PER_GROUP,
                min_samples_per_group=MIN_SAMPLES_PER_GROUP,
                pval_adj_cutoff=PVAL_ADJ_CUTOFF,
                min_logfc=MIN_LOGFC,
                method="mannwhitneyu",
                output_dir=None,
            )
        except Exception as e:
            print(f"Skip fold {fold_id} at train_size={train_size}: marker discovery failed: {repr(e)}")
            continue

        n_marker_genes = int(sum(len(v) for v in marker_dict_fold.values()))
        n_marker_celltypes = int(len(marker_dict_fold))

        marker_count_rows.append({
            "train_size": train_size,
            "train_percent": int(train_size * 100),
            "fold": fold_id,
            "n_marker_celltypes": n_marker_celltypes,
            "n_marker_genes_total": n_marker_genes,
            "marker_dict_json": json.dumps(marker_dict_fold, ensure_ascii=False),
        })

        # 4.2 用 fold 内 marker 分别生成 train/test 特征
        X_basic_train = build_basic_features_for_samples(
            adata_train_fold, train_samples, SAMPLE_COL, CELLTYPE_COL
        )
        X_basic_test = build_basic_features_for_samples(
            adata_test_fold, test_samples, SAMPLE_COL, CELLTYPE_COL, ref_columns=X_basic_train.columns
        )

        X_pb_train = build_pseudobulk_features(
            adata_train_fold, marker_dict_fold, SAMPLE_COL, CELLTYPE_COL
        )
        X_pb_test = build_pseudobulk_features(
            adata_test_fold, marker_dict_fold, SAMPLE_COL, CELLTYPE_COL
        )

        X_pb_train.index = X_pb_train.index.astype(str)
        X_pb_test.index = X_pb_test.index.astype(str)

        X_pb_train = X_pb_train.reindex(index=train_samples, fill_value=0)
        X_pb_test = X_pb_test.reindex(index=test_samples, fill_value=0)
        X_pb_test = X_pb_test.reindex(columns=X_pb_train.columns, fill_value=0)

        X_train = pd.concat([X_basic_train, X_pb_train], axis=1)
        X_test = pd.concat([X_basic_test, X_pb_test], axis=1)
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        # 4.3 建模
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=1.0,
                class_weight="balanced",
                max_iter=1000,
                random_state=random_state
            ))
        ])

        model.fit(X_train, y_train)

        prob_unstable = model.predict_proba(X_test)[:, 1]
        y_pred = (prob_unstable >= threshold).astype(int)

        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, prob_unstable) if len(np.unique(y_test)) == 2 else np.nan

        fold_metrics.append({
            "train_size": train_size,
            "train_percent": int(train_size * 100),
            "fold": fold_id,
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": X_train.shape[1],
            "n_marker_celltypes": n_marker_celltypes,
            "n_marker_genes_total": n_marker_genes,
            "accuracy": acc,
            "precision_unstable": precision,
            "recall_unstable": recall,
            "f1_unstable": f1,
            "roc_auc": auc,
        })

        fold_pred_df = pd.DataFrame({
            "train_size": train_size,
            "train_percent": int(train_size * 100),
            "fold": fold_id,
            "sample_id": X_test.index.astype(str),
            "true_label_numeric": y_test.values,
            "prob_unstable": prob_unstable,
            "pred_label_numeric": y_pred,
        })

        fold_pred_df["true_label"] = fold_pred_df["true_label_numeric"].map({
            1: "unstable",
            0: "stable",
        })
        fold_pred_df["pred_label"] = fold_pred_df["pred_label_numeric"].map({
            1: "unstable",
            0: "stable",
        })

        all_fold_predictions.append(fold_pred_df)

        print(
            f"Accuracy={acc:.4f}, Precision={precision:.4f}, "
            f"Recall={recall:.4f}, F1={f1:.4f}, AUC={auc:.4f}, "
            f"markers={n_marker_genes}"
        )
        print("Confusion matrix:")
        print(confusion_matrix(y_test, y_pred))

    if len(fold_metrics) == 0:
        print(f"No valid folds for train_size={train_size}; skip summary.")
        continue

    fold_metrics_df = pd.DataFrame(fold_metrics)

    summary_rows.append({
        "train_size": train_size,
        "train_percent": int(train_size * 100),
        "n_splits_requested": n_splits,
        "n_splits_valid": fold_metrics_df.shape[0],

        "mean_n_features": fold_metrics_df["n_features"].mean(),
        "mean_n_marker_celltypes": fold_metrics_df["n_marker_celltypes"].mean(),
        "mean_n_marker_genes_total": fold_metrics_df["n_marker_genes_total"].mean(),

        "mean_accuracy": fold_metrics_df["accuracy"].mean(),
        "std_accuracy": fold_metrics_df["accuracy"].std(),

        "mean_precision_unstable": fold_metrics_df["precision_unstable"].mean(),
        "std_precision_unstable": fold_metrics_df["precision_unstable"].std(),

        "mean_recall_unstable": fold_metrics_df["recall_unstable"].mean(),
        "std_recall_unstable": fold_metrics_df["recall_unstable"].std(),

        "mean_f1_unstable": fold_metrics_df["f1_unstable"].mean(),
        "std_f1_unstable": fold_metrics_df["f1_unstable"].std(),

        "mean_roc_auc": fold_metrics_df["roc_auc"].mean(),
        "std_roc_auc": fold_metrics_df["roc_auc"].std(),
    })

# -------------------------
# 5. 保存严格 CV 结果
# -------------------------
summary_df = pd.DataFrame(summary_rows).sort_values("train_size")
all_predictions_df = pd.concat(all_fold_predictions, axis=0, ignore_index=True) if all_fold_predictions else pd.DataFrame()
marker_count_df = pd.DataFrame(marker_count_rows)

summary_df.to_csv(CV_SUMMARY_OUT, index=False)
all_predictions_df.to_csv(CV_PRED_OUT, index=False)
marker_count_df.to_csv(CV_MARKER_COUNT_OUT, index=False)

print("\n" + "=" * 90)
print("Strict fold-wise-marker CV summary")
print("=" * 90)
print(summary_df)

print("\nSaved:")
print(CV_SUMMARY_OUT)
print(CV_PRED_OUT)
print(CV_MARKER_COUNT_OUT)


Strict CV output dir: ./output_auto/human
n_splits: 5
threshold: 0.5
Reading h5ad...
adata shape after filtering unknown cell type: (1004247, 28868)
sample label distribution:
Plaque_type_clean
unknown     790610
unstable    152880
stable       60757
Name: count, dtype: int64
Number of labeled samples: 23
y distribution:
label
1    16
0     7
Name: count, dtype: int64

Strict CV: train_size = 60%, test_size = 40%

Fold 1: n_train=13, n_test=10

===== auto marker discovery: sample-level pseudobulk unstable vs stable within each cell type =====
candidate_celltypes: ['Macrophage', 'Monocyte', 'Dendritic cell', 'Smooth muscle cell', 'Endothelial cell', 'Fibroblast', 'T cell']
DE method: mannwhitneyu

Macrophage: n_cells stable=6080, unstable=5735; n_samples stable=4, unstable=8
Selected 20 sample-level pseudobulk markers for Macrophage using rule: fallback sample-level pseudobulk mannwhitneyu: unstable_minus_stable > 0.0, sorted by p-value/effect
['ELF1', 'TP53BP2', 'PHF19', 'FYN', 'FLT1',

In [ ]:
# ============================================================
# 画严格 CV 折线图
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = "./output_auto/human"
df = pd.read_csv(f"{OUT_DIR}/lr_train_size_cv_summary_foldwise_marker.csv")

df = df.dropna(subset=[
    "train_percent",
    "mean_accuracy",
    "mean_precision_unstable",
    "mean_recall_unstable",
    "mean_f1_unstable",
    "mean_roc_auc",
])

df = df.sort_values("train_percent")

plt.figure(figsize=(8, 5))

plt.errorbar(
    df["train_percent"],
    df["mean_accuracy"],
    yerr=df["std_accuracy"],
    marker="o",
    capsize=4,
    label="Accuracy"
)

plt.errorbar(
    df["train_percent"],
    df["mean_precision_unstable"],
    yerr=df["std_precision_unstable"],
    marker="^",
    capsize=4,
    label="Precision unstable"
)

plt.errorbar(
    df["train_percent"],
    df["mean_recall_unstable"],
    yerr=df["std_recall_unstable"],
    marker="v",
    capsize=4,
    label="Recall unstable"
)

plt.errorbar(
    df["train_percent"],
    df["mean_f1_unstable"],
    yerr=df["std_f1_unstable"],
    marker="s",
    capsize=4,
    label="F1 unstable"
)

plt.errorbar(
    df["train_percent"],
    df["mean_roc_auc"],
    yerr=df["std_roc_auc"],
    marker="D",
    capsize=4,
    label="ROC AUC"
)

plt.xlabel("Training set proportion (%)")
plt.ylabel("Score")
plt.title("Strict CV performance: fold-wise marker selection")
plt.xticks(df["train_percent"])
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


# 最终预测新数据 / unknown 样本

上面的严格交叉验证部分只是为了回答一个问题：**模型在完全未参与 marker 筛选的测试样本上是否稳定**。

真正要预测 unknown / 新数据时，不再留出测试集，而是：

```text
用全部已知 stable / unstable 样本重新找最终 marker
        ↓
用全部已知 stable / unstable 样本生成最终训练特征
        ↓
用全部已知 stable / unstable 样本训练最终 LogisticRegression
        ↓
把 unknown / 新数据整理成相同特征列
        ↓
输出每个 sample 的 unstable 概率和预测类别
```

所以你问的关键点是对的：

- **评估模型时**：每一折只能用 train samples 找 marker，不能用全部 stable / unstable。
- **最终训练并预测 unknown 时**：应该用全部 stable / unstable 样本训练最终模型，这样能利用所有已知标签信息。

注意：不同比例训练集只是用来评估模型稳定性，不代表最终预测时只能用某个比例的样本。


##### mouse

###### homologous

In [13]:
########################同源基因转换#########################################
NEW_H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse.h5ad"  # 只有 PREDICT_MODE = "new_h5ad" 时才需要改这里
adata_new = sc.read_h5ad(NEW_H5AD_PATH)

In [14]:
adata_new.var

,original_gene_names
ENSMUSG00000109644,0610005C13Rik
ENSMUSG00000108652,0610006L08Rik
ENSMUSG00000086714,0610009E02Rik
ENSMUSG00000043644,0610009L18Rik
ENSMUSG00000020831,0610010K14Rik
...,...
ENSMUSG00000119837,n-R5s71
ENSMUSG00000119155,n-R5s86
ENSMUSG00000065107,n-R5s88
ENSMUSG00000119822,n-R5s92


In [15]:
adata_new.var["ensemble_id"] = adata_new.var_names.astype(str)
# 如果 var 里有 original_gene_names，就优先认为它是 mouse gene name
# 否则用当前 var_names
if "original_gene_names" in adata_new.var.columns:
    adata_new.var["mouse_original_gene_names"] = adata_new.var["original_gene_names"].astype(str)
else:
    adata_new.var["mouse_original_gene_names"] = adata_new.var_names.astype(str)

adata_new.var["mouse_original_gene_names"] = adata_new.var["mouse_original_gene_names"].str.strip()

In [16]:
adata_new.var

,original_gene_names,ensemble_id,mouse_original_gene_names
ENSMUSG00000109644,0610005C13Rik,ENSMUSG00000109644,0610005C13Rik
ENSMUSG00000108652,0610006L08Rik,ENSMUSG00000108652,0610006L08Rik
ENSMUSG00000086714,0610009E02Rik,ENSMUSG00000086714,0610009E02Rik
ENSMUSG00000043644,0610009L18Rik,ENSMUSG00000043644,0610009L18Rik
ENSMUSG00000020831,0610010K14Rik,ENSMUSG00000020831,0610010K14Rik
...,...,...,...
ENSMUSG00000119837,n-R5s71,ENSMUSG00000119837,n-R5s71
ENSMUSG00000119155,n-R5s86,ENSMUSG00000119155,n-R5s86
ENSMUSG00000065107,n-R5s88,ENSMUSG00000065107,n-R5s88
ENSMUSG00000119822,n-R5s92,ENSMUSG00000119822,n-R5s92


In [17]:
GENE_MAPPING_PATH = "/home/lixiangyu/zr/Annotate/人类小鼠基因转换表.txt"

MOUSE_ID_COL = "Mouse gene stable ID"
HUMAN_ID_COL = "Gene stable ID"

gene_map = pd.read_csv(GENE_MAPPING_PATH, sep="\t")

# 检查列名
required_cols = [MOUSE_ID_COL, HUMAN_ID_COL]
missing_cols = [c for c in required_cols if c not in gene_map.columns]
if len(missing_cols) > 0:
    raise ValueError(
        f"转换表缺少列: {missing_cols}\n"
        f"当前转换表列名为: {gene_map.columns.tolist()}"
    )

# 清理 ID
gene_map[MOUSE_ID_COL] = gene_map[MOUSE_ID_COL].astype(str).str.strip()
gene_map[HUMAN_ID_COL] = gene_map[HUMAN_ID_COL].astype(str).str.strip()

gene_map = gene_map.replace({"": pd.NA})
gene_map = gene_map.dropna(subset=[MOUSE_ID_COL, HUMAN_ID_COL])

In [18]:
# 如果一个 mouse stable ID 对应多个 human stable ID，保留第一个
mouse_to_human_id = (
    gene_map
    .drop_duplicates(subset=[MOUSE_ID_COL])
    .set_index(MOUSE_ID_COL)[HUMAN_ID_COL]
    .to_dict()
)

# 映射 mouse var_names -> human stable ID
mapped_human_id = adata_new.var["ensemble_id"].map(mouse_to_human_id)

# 新增人类 stable ID 列
adata_new.var["human_ensemble_id"] = mapped_human_id

# 统计
n_total = adata_new.n_vars
n_mapped = mapped_human_id.notna().sum()
n_unmapped = n_total - n_mapped

print(f"Total genes: {n_total}")
print(f"Mapped genes by stable ID: {n_mapped}")
print(f"Unmapped genes: {n_unmapped}")
print(f"Mapping rate: {n_mapped / n_total:.2%}")

# 查看结果
print(
    adata_new.var[
        ["ensemble_id", "human_ensemble_id"]
    ].head(20)
)

Total genes: 57692
Mapped genes by stable ID: 19999
Unmapped genes: 37693
Mapping rate: 34.67%
                           ensemble_id human_ensemble_id
ENSMUSG00000109644  ENSMUSG00000109644               NaN
ENSMUSG00000108652  ENSMUSG00000108652               NaN
ENSMUSG00000086714  ENSMUSG00000086714               NaN
ENSMUSG00000043644  ENSMUSG00000043644               NaN
ENSMUSG00000020831  ENSMUSG00000020831   ENSG00000258315
ENSMUSG00000089755  ENSMUSG00000089755               NaN
ENSMUSG00000046683  ENSMUSG00000046683               NaN
ENSMUSG00000058706  ENSMUSG00000058706   ENSG00000168887
ENSMUSG00000099146  ENSMUSG00000099146               NaN
ENSMUSG00000108236  ENSMUSG00000108236               NaN
ENSMUSG00000097882  ENSMUSG00000097882               NaN
ENSMUSG00000058812  ENSMUSG00000058812               NaN
ENSMUSG00000089889  ENSMUSG00000089889               NaN
ENSMUSG00000087341  ENSMUSG00000087341               NaN
ENSMUSG00000060512  ENSMUSG00000060512   ENSG00000

In [21]:
GENE_NAME_MAPPING_PATH = "/home/lixiangyu/zr/Annotate/gene_names_to_ensembl_ALLFOUND_allfernandez_no6_withallslysz.csv"

HUMAN_ID_IN_ADATA = "human_ensemble_id"
HUMAN_ID_IN_TABLE = "ensembl_id"
HUMAN_GENE_NAME_COL = "gene_name"

# 读取 human gene name 映射表
human_gene_map = pd.read_csv(GENE_NAME_MAPPING_PATH)

# 检查列名
required_cols = [HUMAN_ID_IN_TABLE, HUMAN_GENE_NAME_COL]
missing_cols = [c for c in required_cols if c not in human_gene_map.columns]

if len(missing_cols) > 0:
    raise ValueError(
        f"映射表缺少列: {missing_cols}\n"
        f"当前映射表列名为: {human_gene_map.columns.tolist()}"
    )

# 清理字符串
human_gene_map[HUMAN_ID_IN_TABLE] = (
    human_gene_map[HUMAN_ID_IN_TABLE]
    .astype(str)
    .str.strip()
)

human_gene_map[HUMAN_GENE_NAME_COL] = (
    human_gene_map[HUMAN_GENE_NAME_COL]
    .astype(str)
    .str.strip()
)

adata_new.var[HUMAN_ID_IN_ADATA] = (
    adata_new.var[HUMAN_ID_IN_ADATA]
    .astype("string")
    .str.strip()
)

# 去掉空值
human_gene_map = human_gene_map.replace({"": pd.NA})
human_gene_map = human_gene_map.dropna(
    subset=[HUMAN_ID_IN_TABLE, HUMAN_GENE_NAME_COL]
)

# 如果一个 ensembl_id 对应多个 Human gene name，保留第一个
human_id_to_gene_name = (
    human_gene_map
    .drop_duplicates(subset=[HUMAN_ID_IN_TABLE])
    .set_index(HUMAN_ID_IN_TABLE)[HUMAN_GENE_NAME_COL]
    .to_dict()
)

# 映射 human_ensemble_id -> Human gene name
mapped_human_gene_name = adata_new.var[HUMAN_ID_IN_ADATA].map(
    human_id_to_gene_name
)

# 新增 human_gene_name 列
# 没匹配上的保留为空值 NaN
adata_new.var["human_gene_name"] = mapped_human_gene_name

# 统计
n_total = adata_new.n_vars
n_mapped = mapped_human_gene_name.notna().sum()
n_unmapped = n_total - n_mapped

print(f"Total genes: {n_total}")
print(f"Mapped human gene names: {n_mapped}")
print(f"Unmapped human gene names: {n_unmapped}")
print(f"Mapping rate: {n_mapped / n_total:.2%}")

# 查看结果
print(
    adata_new.var[
        [HUMAN_ID_IN_ADATA, "human_gene_name"]
    ].head(20)
)

Total genes: 57692
Mapped human gene names: 18652
Unmapped human gene names: 39040
Mapping rate: 32.33%
                   human_ensemble_id human_gene_name
ENSMUSG00000109644              <NA>             NaN
ENSMUSG00000108652              <NA>             NaN
ENSMUSG00000086714              <NA>             NaN
ENSMUSG00000043644              <NA>             NaN
ENSMUSG00000020831   ENSG00000258315        C17orf49
ENSMUSG00000089755              <NA>             NaN
ENSMUSG00000046683              <NA>             NaN
ENSMUSG00000058706   ENSG00000168887         C2orf68
ENSMUSG00000099146              <NA>             NaN
ENSMUSG00000108236              <NA>             NaN
ENSMUSG00000097882              <NA>             NaN
ENSMUSG00000058812              <NA>             NaN
ENSMUSG00000089889              <NA>             NaN
ENSMUSG00000087341              <NA>             NaN
ENSMUSG00000060512   ENSG00000154274         C4orf19
ENSMUSG00000087361              <NA>            

In [22]:
adata_new.var

,original_gene_names,ensemble_id,mouse_original_gene_names,human_ensemble_id,human_gene_name
ENSMUSG00000109644,0610005C13Rik,ENSMUSG00000109644,0610005C13Rik,<NA>,NaN
ENSMUSG00000108652,0610006L08Rik,ENSMUSG00000108652,0610006L08Rik,<NA>,NaN
ENSMUSG00000086714,0610009E02Rik,ENSMUSG00000086714,0610009E02Rik,<NA>,NaN
ENSMUSG00000043644,0610009L18Rik,ENSMUSG00000043644,0610009L18Rik,<NA>,NaN
ENSMUSG00000020831,0610010K14Rik,ENSMUSG00000020831,0610010K14Rik,ENSG00000258315,C17orf49
...,...,...,...,...,...
ENSMUSG00000119837,n-R5s71,ENSMUSG00000119837,n-R5s71,<NA>,NaN
ENSMUSG00000119155,n-R5s86,ENSMUSG00000119155,n-R5s86,<NA>,NaN
ENSMUSG00000065107,n-R5s88,ENSMUSG00000065107,n-R5s88,<NA>,NaN
ENSMUSG00000119822,n-R5s92,ENSMUSG00000119822,n-R5s92,<NA>,NaN


In [25]:
adata_new.var_names = adata_new.var['human_gene_name'].values
adata_new.var

,original_gene_names,ensemble_id,mouse_original_gene_names,human_ensemble_id,human_gene_name
NaN,0610005C13Rik,ENSMUSG00000109644,0610005C13Rik,<NA>,NaN
NaN,0610006L08Rik,ENSMUSG00000108652,0610006L08Rik,<NA>,NaN
NaN,0610009E02Rik,ENSMUSG00000086714,0610009E02Rik,<NA>,NaN
NaN,0610009L18Rik,ENSMUSG00000043644,0610009L18Rik,<NA>,NaN
C17orf49,0610010K14Rik,ENSMUSG00000020831,0610010K14Rik,ENSG00000258315,C17orf49
...,...,...,...,...,...
NaN,n-R5s71,ENSMUSG00000119837,n-R5s71,<NA>,NaN
NaN,n-R5s86,ENSMUSG00000119155,n-R5s86,<NA>,NaN
NaN,n-R5s88,ENSMUSG00000065107,n-R5s88,<NA>,NaN
NaN,n-R5s92,ENSMUSG00000119822,n-R5s92,<NA>,NaN


In [27]:
adata_new.var_names = adata_new.var_names.fillna("nan").astype(str)
adata_new.var_names_make_unique()

In [28]:
adata_new.var

,original_gene_names,ensemble_id,mouse_original_gene_names,human_ensemble_id,human_gene_name
nan,0610005C13Rik,ENSMUSG00000109644,0610005C13Rik,<NA>,NaN
nan-1,0610006L08Rik,ENSMUSG00000108652,0610006L08Rik,<NA>,NaN
nan-2,0610009E02Rik,ENSMUSG00000086714,0610009E02Rik,<NA>,NaN
nan-3,0610009L18Rik,ENSMUSG00000043644,0610009L18Rik,<NA>,NaN
C17orf49,0610010K14Rik,ENSMUSG00000020831,0610010K14Rik,ENSG00000258315,C17orf49
...,...,...,...,...,...
nan-39035,n-R5s71,ENSMUSG00000119837,n-R5s71,<NA>,NaN
nan-39036,n-R5s86,ENSMUSG00000119155,n-R5s86,<NA>,NaN
nan-39037,n-R5s88,ENSMUSG00000065107,n-R5s88,<NA>,NaN
nan-39038,n-R5s92,ENSMUSG00000119822,n-R5s92,<NA>,NaN


In [ ]:
adata_new.var['human_gene_name'] = adata_new.var_names
adata_new.var

,original_gene_names,ensemble_id,mouse_original_gene_names,human_ensemble_id,human_gene_name
nan,0610005C13Rik,ENSMUSG00000109644,0610005C13Rik,<NA>,nan
nan-1,0610006L08Rik,ENSMUSG00000108652,0610006L08Rik,<NA>,nan-1
nan-2,0610009E02Rik,ENSMUSG00000086714,0610009E02Rik,<NA>,nan-2
nan-3,0610009L18Rik,ENSMUSG00000043644,0610009L18Rik,<NA>,nan-3
C17orf49,0610010K14Rik,ENSMUSG00000020831,0610010K14Rik,ENSG00000258315,C17orf49
...,...,...,...,...,...
nan-39035,n-R5s71,ENSMUSG00000119837,n-R5s71,<NA>,nan-39035
nan-39036,n-R5s86,ENSMUSG00000119155,n-R5s86,<NA>,nan-39036
nan-39037,n-R5s88,ENSMUSG00000065107,n-R5s88,<NA>,nan-39037
nan-39038,n-R5s92,ENSMUSG00000119822,n-R5s92,<NA>,nan-39038


In [34]:
adata_new.var_names = adata_new.var['human_ensemble_id'].values
adata_new.var_names = adata_new.var_names.fillna("nan").astype(str)
adata_new.var_names_make_unique()
adata_new.var
adata_new.var['human_ensemble_id'] = adata_new.var_names
adata_new.var_names = adata_new.var['human_gene_name'].values
adata_new.var

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:947: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    [<NA>, <NA>, <NA>, <NA>, 'ENSG00000258315']

    Inferred to be: categorical

  names = self._prep_dim_index(names, "var")


,original_gene_names,ensemble_id,mouse_original_gene_names,human_ensemble_id,human_gene_name
nan,0610005C13Rik,ENSMUSG00000109644,0610005C13Rik,nan,nan
nan-1,0610006L08Rik,ENSMUSG00000108652,0610006L08Rik,nan-1,nan-1
nan-2,0610009E02Rik,ENSMUSG00000086714,0610009E02Rik,nan-2,nan-2
nan-3,0610009L18Rik,ENSMUSG00000043644,0610009L18Rik,nan-3,nan-3
C17orf49,0610010K14Rik,ENSMUSG00000020831,0610010K14Rik,ENSG00000258315,C17orf49
...,...,...,...,...,...
nan-39035,n-R5s71,ENSMUSG00000119837,n-R5s71,nan-37688,nan-39035
nan-39036,n-R5s86,ENSMUSG00000119155,n-R5s86,nan-37689,nan-39036
nan-39037,n-R5s88,ENSMUSG00000065107,n-R5s88,nan-37690,nan-39037
nan-39038,n-R5s92,ENSMUSG00000119822,n-R5s92,nan-37691,nan-39038


In [35]:
adata_new

AnnData object with n_obs × n_vars = 564966 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2', 'cell_type_level3'
    var: 'original_gene_names', 'ensemble_id', 'mouse_original_gene_names', 'human_ensemble_id', 'human_gene_name'
    uns: 'cell_type_level1_corrected_colors', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UM

In [36]:
NEW_H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse_homologous.h5ad"  # 只有 PREDICT_MODE = "new_h5ad" 时才需要改这里
adata_new.write_h5ad(NEW_H5AD_PATH)

###### pre

In [4]:
# ============================================================
# 最终模型训练 + 预测 unknown / 新 h5ad 数据
# ============================================================
# 用法：
# 1. 如果只是预测前面已经从同一个 h5ad 里生成的 unknown 样本：
#       PREDICT_MODE = "existing_unknown"
#
# 2. 如果要预测一个全新的 h5ad 文件：
#       PREDICT_MODE = "new_h5ad"
#       NEW_H5AD_PATH = "/your/path/to/new_data.h5ad"
#
# 前提：
# - 前面的 prepare input cell 已经运行过，已经生成训练特征文件：
#   ./output_auto/X_combined_basic_plus_pseudobulk.csv
#   ./output_auto/y_sample_label_numeric.csv
# - 如果使用 existing_unknown，还需要已经生成：
#   ./output_auto/X_unknown_combined_basic_plus_pseudobulk.csv
# - 如果使用 new_h5ad，新数据里需要有 SAMPLE_COL 和 CELLTYPE_COL 对应的 obs 列。
# ============================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# -------------------------
# 1. 参数设置
# -------------------------
PREDICT_MODE = "new_h5ad"   # 可改成 "new_h5ad"
NEW_H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse_homologous.h5ad"  # 只有 PREDICT_MODE = "new_h5ad" 时才需要改这里
OUT_DIR = "./output_auto/mouse"
PRED_THRESHOLD = 0.5        # 判定 unstable-like 的阈值；如果你想更保守，可以改成 0.6
HIGH_CONF_THRESHOLD = 0.8   # 高置信度 unstable 阈值

FINAL_PRED_OUT = f"{OUT_DIR}/final_prediction_unstable_probability.csv"
FINAL_COEF_OUT = f"{OUT_DIR}/final_lr_feature_coefficients.csv"
OUT_X_UNKNOWN_COMBINED = f"{OUT_DIR}/X_unknown_combined_basic_plus_pseudobulk.csv"

os.makedirs(OUT_DIR, exist_ok=True)
AUTO_MARKER_DICT_PATH = f"./output_auto/human/auto_marker_dict.json"
if "marker_dict" not in globals():
    marker_dict = load_marker_dict(AUTO_MARKER_DICT_PATH)
    print(f"Loaded marker_dict from: {AUTO_MARKER_DICT_PATH}")


SAMPLE_COL = "sample"
# CELLTYPE_COL = "cell_type_level1"
CELLTYPE_COL = "cell_type_level1_corrected"##应该用cell_type_level1_corrected
LABEL_RAW_COL = "Plaque_type"
LABEL_COL = "Plaque_type_clean"

# -------------------------
# 2. 读取训练特征和标签
# -------------------------
X_train = pd.read_csv(f"./output_auto/human/X_combined_basic_plus_pseudobulk.csv", index_col=0)
y_train = pd.read_csv(f"./output_auto/human/y_sample_label_numeric.csv", index_col=0).iloc[:, 0]

# 确保 X 和 y 顺序一致
X_train = X_train.loc[y_train.index]

print("Training X shape:", X_train.shape)
print("Training y distribution:")
print(y_train.value_counts())

# -------------------------
# 3. 用全部已标注样本训练最终模型
# -------------------------
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)
print("\nFinal model fitted using all labeled stable / unstable samples.")

# -------------------------
# 4A. 情况一：预测前面已经生成好的 unknown 特征
# -------------------------
if PREDICT_MODE == "existing_unknown":
    X_pred = pd.read_csv(OUT_X_UNKNOWN_COMBINED, index_col=0)
    X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)
    pred_source = "existing_unknown"

# -------------------------
# 4B. 情况二：预测一个新的 h5ad 文件
# -------------------------
elif PREDICT_MODE == "new_h5ad":
    if not os.path.exists(NEW_H5AD_PATH):
        raise FileNotFoundError(
            f"找不到 NEW_H5AD_PATH: {NEW_H5AD_PATH}\n"
            "请把 NEW_H5AD_PATH 改成你的新 h5ad 文件路径。"
        )

    print("\nReading new h5ad...")
    adata_new = sc.read_h5ad(NEW_H5AD_PATH)

    # 和训练数据保持一致：去掉未知 cell type
    if CELLTYPE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 CELLTYPE_COL: {CELLTYPE_COL}")
    if SAMPLE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 SAMPLE_COL: {SAMPLE_COL}")

    adata_new = adata_new[adata_new.obs[CELLTYPE_COL] != "unknown"].copy()

    # 如果新数据里也有 original_gene_names，就按你前面训练时的方式改 var_names
    # ============================================================
    # 预测 mouse homologous h5ad 时，强制使用 human_gene_name 作为 var_names
    # ============================================================

    if "human_gene_name" not in adata_new.var.columns:
        raise ValueError("adata_new.var 里没有 human_gene_name 列，请先完成 mouse -> human gene name 转换。")

    adata_new.var["original_feature_id"] = adata_new.var_names.astype(str)

    adata_new.var["human_gene_name"] = (
        adata_new.var["human_gene_name"]
        .astype("string")
        .fillna("nan")
        .astype(str)
        .str.strip()
    )

    adata_new.var["human_gene_name"] = adata_new.var["human_gene_name"].replace("", "nan")

    adata_new.var_names = adata_new.var["human_gene_name"].astype(str)
    adata_new.var_names_make_unique()

    print("Using human_gene_name as adata_new.var_names")
    print("Example var_names:")
    print(adata_new.var_names[:20])

    print("New adata shape after filtering unknown cell type:", adata_new.shape)

    # 生成新数据的 cell type proportion 特征
    obs_new = adata_new.obs[[SAMPLE_COL, CELLTYPE_COL]].copy()
    obs_new = obs_new.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])

    cell_count_new = pd.crosstab(obs_new[SAMPLE_COL], obs_new[CELLTYPE_COL])
    cell_prop_new = cell_count_new.div(cell_count_new.sum(axis=1), axis=0)
    cell_prop_new.columns = [f"prop__{c}" for c in cell_prop_new.columns]

    # 读取训练时的 basic / pseudobulk 列，保证新数据列顺序完全一致
    X_basic_train = pd.read_csv(f"./output_auto/human/X_basic_celltype_proportion.csv", index_col=0)
    X_pb_train = pd.read_csv(f"./output_auto/human/X_pseudobulk_marker.csv", index_col=0)

    X_basic_new = cell_prop_new.reindex(columns=X_basic_train.columns, fill_value=0)

    # 生成新数据的 pseudobulk marker 特征
    X_pb_new = build_pseudobulk_features(
        adata_new,
        marker_dict,
        SAMPLE_COL,
        CELLTYPE_COL
    )
    X_pb_new = X_pb_new.reindex(columns=X_pb_train.columns, fill_value=0)

    # 拼接并对齐到最终训练特征列
    common_new_samples = X_basic_new.index.intersection(X_pb_new.index)
    X_pred = pd.concat([
        X_basic_new.loc[common_new_samples],
        X_pb_new.loc[common_new_samples]
    ], axis=1)

    X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)
    pred_source = "new_h5ad"

else:
    raise ValueError('PREDICT_MODE 只能是 "existing_unknown" 或 "new_h5ad"')

print("\nPrediction X shape:", X_pred.shape)
print("Prediction source:", pred_source)

# -------------------------
# 5. 预测 unstable 概率
# -------------------------
prob_unstable = final_model.predict_proba(X_pred)[:, 1]

pred_df = pd.DataFrame({
    "sample_id": X_pred.index,
    "prob_unstable": prob_unstable
})

# 二分类结果
pred_df[f"pred_label_threshold_{PRED_THRESHOLD}"] = np.where(
    pred_df["prob_unstable"] >= PRED_THRESHOLD,
    "unstable-like",
    "non-unstable"
)

# 三档结果，方便生物学解释
def assign_3class(p):
    if p >= HIGH_CONF_THRESHOLD:
        return "high-confidence unstable"
    elif p >= PRED_THRESHOLD:
        return "unstable-like"
    else:
        return "non-unstable"

pred_df["pred_3class"] = pred_df["prob_unstable"].apply(assign_3class)

# 按 unstable 概率从高到低排序
pred_df = pred_df.sort_values("prob_unstable", ascending=False)

print("\nPrediction result:")
print(pred_df)

pred_df.to_csv(FINAL_PRED_OUT, index=False)
print(f"\nSaved prediction result: {FINAL_PRED_OUT}")

# -------------------------
# 6. 导出 LogisticRegression 特征系数，方便看模型主要依据什么判断 unstable
# -------------------------
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": final_model.named_steps["clf"].coef_[0]
})

coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

coef_df.to_csv(FINAL_COEF_OUT, index=False)
print(f"Saved feature coefficients: {FINAL_COEF_OUT}")

print("\nTop positive features: higher value pushes prediction toward unstable")
print(coef_df.sort_values("coef", ascending=False).head(20))

print("\nTop negative features: higher value pushes prediction toward non-unstable")
print(coef_df.sort_values("coef", ascending=True).head(20))


Training X shape: (23, 160)
Training y distribution:
label
1    16
0     7
Name: count, dtype: int64

Final model fitted using all labeled stable / unstable samples.

Reading new h5ad...
Using human_gene_name as adata_new.var_names
Example var_names:
Index(['nan', 'nan-1', 'nan-2', 'nan-3', 'C17orf49', 'nan-4', 'nan-5',
       'C2orf68', 'nan-6', 'nan-7', 'nan-8', 'nan-9', 'nan-10', 'nan-11',
       'C4orf19', 'nan-12', 'nan-13', 'nan-14', 'nan-15', 'RP11-766F14.2'],
      dtype='object', name='human_gene_name')
New adata shape after filtering unknown cell type: (564966, 57692)

===== marker presence check =====
Macrophage:
  present: ['CIDEB', 'RPL18A', 'DGKD', 'CACNA2D4', 'PHLDB1', 'TM4SF20', 'DDIT4', 'PTPRE', 'NFKB2', 'RELB', 'AEN', 'S1PR3', 'RPS9']
  missing: ['ASMTL', 'MIR3945HG', 'AL358472.4', 'AC123512.1', 'AC007384.1', 'AC132872.5', 'AC092053.4']
Monocyte:
  present: ['CTH', 'PLOD2', 'PLEKHA4', 'MYOM1', 'TMEM132A', 'CITED4', 'GPRC5A', 'SPATA2L', 'SPRY1']
  missing: ['AC083837.1

In [6]:
import pandas as pd

pred_path = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/10_downstream/Infer_plaque_type/0523_no_Basophil/output_auto/mouse/final_prediction_unstable_probability.csv"
pred = pd.read_csv(pred_path)
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse.h5ad")
# 只保留需要的列，并改名
pred_sub = pred[[
    "sample_id",
    "pred_label_threshold_0.5",
    "pred_3class"
]].rename(columns={
    "pred_label_threshold_0.5": "Plaque_type_pred",
    "pred_3class": "Plaque_type_pred_3class"
})

# 用 sample_id 映射到 adata.obs
mapping_1 = pred_sub.set_index("sample_id")["Plaque_type_pred"]
mapping_2 = pred_sub.set_index("sample_id")["Plaque_type_pred_3class"]

adata.obs["Plaque_type_pred"] = adata.obs["sample"].map(mapping_1)
adata.obs["Plaque_type_pred_3class"] = adata.obs["sample"].map(mapping_2)

In [7]:
adata.obs['Plaque_type_pred'].value_counts()

Plaque_type_pred
non-unstable     370677
unstable-like    194289
Name: count, dtype: int64

In [8]:
###下一步转换成rds文件
adata.write("/home/lixiangyu/zr/Annotate/ANNOTATE_new/10_downstream/Infer_plaque_type/0523_no_Basophil/output_auto/mouse/mouse_plaque_type_pred.h5ad")
adata

AnnData object with n_obs × n_vars = 564966 × 57692
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'tissue', 'cell_type_level1', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'atlas_key', 'cell_type_level1_corrected', 'cell_type_level2', 'cell_type_level3', 'Plaque_type_pred', 'Plaque_type_pred_3class'
    var: 'original_gene_names'
    uns: 'cell_type_level1_corrected_colors', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap', 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'de

##### human

In [4]:
# ============================================================
# 最终模型训练 + 预测 unknown / 新 h5ad 数据
# ============================================================
# 用法：
# 1. 如果只是预测前面已经从同一个 h5ad 里生成的 unknown 样本：
#       PREDICT_MODE = "existing_unknown"
#
# 2. 如果要预测一个全新的 h5ad 文件：
#       PREDICT_MODE = "new_h5ad"
#       NEW_H5AD_PATH = "/your/path/to/new_data.h5ad"
#
# 前提：
# - 前面的 prepare input cell 已经运行过，已经生成训练特征文件：
#   ./output_auto/human/X_combined_basic_plus_pseudobulk.csv
#   ./output_auto/human/y_sample_label_numeric.csv
# - 如果使用 existing_unknown，还需要已经生成：
#   ./output_auto/human/X_unknown_combined_basic_plus_pseudobulk.csv
# - 如果使用 new_h5ad，新数据里需要有 SAMPLE_COL 和 CELLTYPE_COL 对应的 obs 列。
# ============================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# -------------------------
# 1. 参数设置
# -------------------------
PREDICT_MODE = "existing_unknown"   # 可改成 "new_h5ad"
# NEW_H5AD_PATH = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_mouse/scPoli_concat_level3_marker_allmouse.h5ad"  # 只有 PREDICT_MODE = "new_h5ad" 时才需要改这里
OUT_DIR = "./output_auto/human"
PRED_THRESHOLD = 0.5        # 判定 unstable-like 的阈值；如果你想更保守，可以改成 0.6
HIGH_CONF_THRESHOLD = 0.8   # 高置信度 unstable 阈值

FINAL_PRED_OUT = f"{OUT_DIR}/final_prediction_unstable_probability.csv"
FINAL_COEF_OUT = f"{OUT_DIR}/final_lr_feature_coefficients.csv"
OUT_X_UNKNOWN_COMBINED = f"{OUT_DIR}/X_unknown_combined_basic_plus_pseudobulk.csv"

os.makedirs(OUT_DIR, exist_ok=True)
AUTO_MARKER_DICT_PATH = f"{OUT_DIR}/auto_marker_dict.json"
if "marker_dict" not in globals():
    marker_dict = load_marker_dict(AUTO_MARKER_DICT_PATH)
    print(f"Loaded marker_dict from: {AUTO_MARKER_DICT_PATH}")


SAMPLE_COL = "sample"
# CELLTYPE_COL = "cell_type_level1"
CELLTYPE_COL = "cell_type_level1_corrected"##应该用cell_type_level1_corrected
LABEL_RAW_COL = "Plaque_type"
LABEL_COL = "Plaque_type_clean"

# -------------------------
# 2. 读取训练特征和标签
# -------------------------
X_train = pd.read_csv(f"{OUT_DIR}/X_combined_basic_plus_pseudobulk.csv", index_col=0)
y_train = pd.read_csv(f"{OUT_DIR}/y_sample_label_numeric.csv", index_col=0).iloc[:, 0]

# 确保 X 和 y 顺序一致
X_train = X_train.loc[y_train.index]

print("Training X shape:", X_train.shape)
print("Training y distribution:")
print(y_train.value_counts())

# -------------------------
# 3. 用全部已标注样本训练最终模型
# -------------------------
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)
print("\nFinal model fitted using all labeled stable / unstable samples.")

# -------------------------
# 4A. 情况一：预测前面已经生成好的 unknown 特征
# -------------------------
if PREDICT_MODE == "existing_unknown":
    X_pred = pd.read_csv(OUT_X_UNKNOWN_COMBINED, index_col=0)
    X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)
    pred_source = "existing_unknown"

# -------------------------
# 4B. 情况二：预测一个新的 h5ad 文件
# -------------------------
elif PREDICT_MODE == "new_h5ad":
    if not os.path.exists(NEW_H5AD_PATH):
        raise FileNotFoundError(
            f"找不到 NEW_H5AD_PATH: {NEW_H5AD_PATH}\n"
            "请把 NEW_H5AD_PATH 改成你的新 h5ad 文件路径。"
        )

    print("\nReading new h5ad...")
    adata_new = sc.read_h5ad(NEW_H5AD_PATH)

    # 和训练数据保持一致：去掉未知 cell type
    if CELLTYPE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 CELLTYPE_COL: {CELLTYPE_COL}")
    if SAMPLE_COL not in adata_new.obs.columns:
        raise ValueError(f"新数据 adata.obs 里缺少 SAMPLE_COL: {SAMPLE_COL}")

    adata_new = adata_new[adata_new.obs[CELLTYPE_COL] != "unknown"].copy()

    # 如果新数据里也有 original_gene_names，就按你前面训练时的方式改 var_names
    if "original_gene_names" in adata_new.var.columns:
        adata_new.var["original_feature_id"] = adata_new.var_names.astype(str)
        adata_new.var_names = adata_new.var["original_gene_names"].astype(str)
        adata_new.var_names_make_unique()

    print("New adata shape after filtering unknown cell type:", adata_new.shape)

    # 生成新数据的 cell type proportion 特征
    obs_new = adata_new.obs[[SAMPLE_COL, CELLTYPE_COL]].copy()
    obs_new = obs_new.dropna(subset=[SAMPLE_COL, CELLTYPE_COL])

    cell_count_new = pd.crosstab(obs_new[SAMPLE_COL], obs_new[CELLTYPE_COL])
    cell_prop_new = cell_count_new.div(cell_count_new.sum(axis=1), axis=0)
    cell_prop_new.columns = [f"prop__{c}" for c in cell_prop_new.columns]

    # 读取训练时的 basic / pseudobulk 列，保证新数据列顺序完全一致
    X_basic_train = pd.read_csv(f"{OUT_DIR}/X_basic_celltype_proportion.csv", index_col=0)
    X_pb_train = pd.read_csv(f"{OUT_DIR}/X_pseudobulk_marker.csv", index_col=0)

    X_basic_new = cell_prop_new.reindex(columns=X_basic_train.columns, fill_value=0)

    # 生成新数据的 pseudobulk marker 特征
    X_pb_new = build_pseudobulk_features(
        adata_new,
        marker_dict,
        SAMPLE_COL,
        CELLTYPE_COL
    )
    X_pb_new = X_pb_new.reindex(columns=X_pb_train.columns, fill_value=0)

    # 拼接并对齐到最终训练特征列
    common_new_samples = X_basic_new.index.intersection(X_pb_new.index)
    X_pred = pd.concat([
        X_basic_new.loc[common_new_samples],
        X_pb_new.loc[common_new_samples]
    ], axis=1)

    X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)
    pred_source = "new_h5ad"

else:
    raise ValueError('PREDICT_MODE 只能是 "existing_unknown" 或 "new_h5ad"')

print("\nPrediction X shape:", X_pred.shape)
print("Prediction source:", pred_source)

# -------------------------
# 5. 预测 unstable 概率
# -------------------------
prob_unstable = final_model.predict_proba(X_pred)[:, 1]

pred_df = pd.DataFrame({
    "sample_id": X_pred.index,
    "prob_unstable": prob_unstable
})

# 二分类结果
pred_df[f"pred_label_threshold_{PRED_THRESHOLD}"] = np.where(
    pred_df["prob_unstable"] >= PRED_THRESHOLD,
    "unstable-like",
    "non-unstable"
)

# 三档结果，方便生物学解释
def assign_3class(p):
    if p >= HIGH_CONF_THRESHOLD:
        return "high-confidence unstable"
    elif p >= PRED_THRESHOLD:
        return "unstable-like"
    else:
        return "non-unstable"

pred_df["pred_3class"] = pred_df["prob_unstable"].apply(assign_3class)

# 按 unstable 概率从高到低排序
pred_df = pred_df.sort_values("prob_unstable", ascending=False)

print("\nPrediction result:")
print(pred_df)

pred_df.to_csv(FINAL_PRED_OUT, index=False)
print(f"\nSaved prediction result: {FINAL_PRED_OUT}")

# -------------------------
# 6. 导出 LogisticRegression 特征系数，方便看模型主要依据什么判断 unstable
# -------------------------
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": final_model.named_steps["clf"].coef_[0]
})

coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

coef_df.to_csv(FINAL_COEF_OUT, index=False)
print(f"Saved feature coefficients: {FINAL_COEF_OUT}")

print("\nTop positive features: higher value pushes prediction toward unstable")
print(coef_df.sort_values("coef", ascending=False).head(20))

print("\nTop negative features: higher value pushes prediction toward non-unstable")
print(coef_df.sort_values("coef", ascending=True).head(20))


Loaded marker_dict from: ./output_auto/human/auto_marker_dict.json
Training X shape: (23, 160)
Training y distribution:
label
1    16
0     7
Name: count, dtype: int64

Final model fitted using all labeled stable / unstable samples.

Prediction X shape: (170, 160)
Prediction source: existing_unknown

Prediction result:
        sample_id  prob_unstable pred_label_threshold_0.5  \
18    GSE143921_2       1.000000            unstable-like   
133  GSE247238_17       1.000000            unstable-like   
21    GSE143921_5       1.000000            unstable-like   
20    GSE143921_4       1.000000            unstable-like   
74    GSE224273_1       1.000000            unstable-like   
..            ...            ...                      ...   
98   GSE226492_10       0.000595             non-unstable   
47    GSE166676_5       0.000530             non-unstable   
157           MXT       0.000505             non-unstable   
160            S2       0.000291             non-unstable   
159     

In [3]:
import pandas as pd

pred_path = "/home/lixiangyu/zr/Annotate/ANNOTATE_new/10_downstream/Infer_plaque_type/0523_no_Basophil/output_auto/human/final_prediction_unstable_probability.csv"
pred = pd.read_csv(pred_path)
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/9_annotate_level3/0521_no_Basophil/output_allhuman/scPoli_concat_level3_marker_all_metadata.h5ad")
# 只保留需要的列，并改名
pred_sub = pred[[
    "sample_id",
    "pred_label_threshold_0.5",
    "pred_3class"
]].rename(columns={
    "pred_label_threshold_0.5": "Plaque_type_pred",
    "pred_3class": "Plaque_type_pred_3class"
})

# 用 sample_id 映射到 adata.obs
mapping_1 = pred_sub.set_index("sample_id")["Plaque_type_pred"]
mapping_2 = pred_sub.set_index("sample_id")["Plaque_type_pred_3class"]

adata.obs["Plaque_type_pred"] = adata.obs["sample"].map(mapping_1)
adata.obs["Plaque_type_pred_3class"] = adata.obs["sample"].map(mapping_2)

In [4]:
adata.obs['Plaque_type_pred'].value_counts()

Plaque_type_pred
unstable-like    479443
non-unstable     311167
Name: count, dtype: int64

In [5]:
###下一步转换成rds文件
adata.write("/home/lixiangyu/zr/Annotate/ANNOTATE_new/10_downstream/Infer_plaque_type/0523_no_Basophil/output_auto/human/human_plaque_type_pred.h5ad")
adata

AnnData object with n_obs × n_vars = 1004247 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2', 'cell_type_level3', 'Plaque_type', 'Plaque_type_pred', 'Plaque_type_pred_3class'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

### end